####Working with dates
1. [Date functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#date-and-timestamp-functions)

####1. Requirement
You are given a dataframe as below

In [0]:
from pyspark.sql.functions import concat_ws

data_list = [(2022, 5, 18) , (9999, 12, 31), (-9999, 1, 1), (10000, 1, 1), 
             (-10000, 1, 1), (193, 5, 25),   (99, 5, 25),   (1000, 2, 29)]

df = (spark.createDataFrame(data_list).toDF("Y", "M", "D")
     .withColumn("date_str", concat_ws("-", "Y", "M", "D")))

df.display()

####2. Convert strings to date

1. Spark validates the date against the Proleptic Gregorian calendar.
2. The negative years are BC, and the positive values are AD in Gregorian calander.
3. Valid dates are taken, and invalid dates throw an exception or taken as null

In [0]:
from pyspark.sql.functions import expr
df = (
    df.withColumn("valid_date", expr("try_to_date(date_str, 'y-M-d')"))
        .drop("Y", "M", "D")
)

df.display()

####3. Add, Subtract days and months to date


In [0]:
from pyspark.sql.functions import date_add, date_sub, add_months, date_diff

df = (
    df.withColumns({
        "add_5_days": date_add("valid_date", 5),
        "sub_5_days": date_sub("valid_date", 5),
        "add_5_months": add_months("valid_date", 5),
        "sub_5_months": add_months("valid_date", -5)
    })
)

df.display()

####4. Current date, date difference, and interval

In [0]:
from pyspark.sql.functions import current_date, date_diff, col
df = (
    df.withColumns({
        "current_date": current_date(),
        "delta_date_days": date_diff("add_5_months", "valid_date"),
        "delta_date_interval": col("current_date") - col("valid_date")
    })
)

df.display()

####5. Format date

In [0]:
from pyspark.sql.functions import date_format

df = (
    df.withColumn("fmt_date", date_format("valid_date", "dd MMM yyyy"))
)

df.display()